In [2]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from sklearn.metrics.cluster import pair_confusion_matrix
import itertools
import numpy as np
import pandas as pd
import geopandas as gpd
from joblib import Parallel, delayed

In [3]:
shapes   = ['square', 'pent', 'hex']
sizes    = [100, 400]
gdfs_reg = ["south", "chicago", "ceara_zika"]

In [4]:
def pair_confusion_counts(true_labels, pred_labels):

    if len(true_labels) <= 1:
        return np.nan, np.nan, np.nan, np.nan
    cm = pair_confusion_matrix(true_labels, pred_labels)
    tn, fn = cm[0, 0], cm[0, 1]
    fp, tp = cm[1, 0], cm[1, 1]
    return tp, fp, fn, tn

In [5]:
def evaluate_group_synthetic(group_key, group_df):

    method, missing_edges, c_run, z_run = group_key
    z_run = int(z_run)
    ward_pred   = group_df["ward_label"].values
    skater_pred = group_df["skater_label"].values
    ward_true   = group_df[f"true_ward_zone{z_run}"].values
    skater_true = group_df[f"true_skater_zone{z_run}"].values

    # ward
    if np.all(ward_pred == -1):
        ari_ward = ami_ward = np.nan
        tp_ward = fp_ward = fn_ward = tn_ward = np.nan
    else:
        valid = (ward_pred != -1) & (ward_true != -1)
        if valid.sum() > 1:
            ari_ward = adjusted_rand_score(ward_true[valid], ward_pred[valid])
            ami_ward = adjusted_mutual_info_score(ward_true[valid], ward_pred[valid])
            tp_ward, fp_ward, fn_ward, tn_ward = pair_confusion_counts(
                ward_true[valid], ward_pred[valid]
            )
        else:
            ari_ward = ami_ward = np.nan
            tp_ward = fp_ward = fn_ward = tn_ward = np.nan

    # skater
    if np.all(skater_pred == -1):
        ari_skater = ami_skater = np.nan
        tp_skater = fp_skater = fn_skater = tn_skater = np.nan
    else:
        valid = (skater_pred != -1) & (skater_true != -1)
        if valid.sum() > 1:
            ari_skater = adjusted_rand_score(skater_true[valid], skater_pred[valid])
            ami_skater = adjusted_mutual_info_score(skater_true[valid], skater_pred[valid])
            tp_skater, fp_skater, fn_skater, tn_skater = pair_confusion_counts(
                skater_true[valid], skater_pred[valid]
            )
        else:
            ari_skater = ami_skater = np.nan
            tp_skater = fp_skater = fn_skater = tn_skater = np.nan

    return {
        "method"    : method,
        "missing_edges" : missing_edges,
        "c_run"     : c_run,
        "z_run"     : z_run,
        "ari_ward"  : ari_ward,
        "ami_ward"  : ami_ward,
        "tp_ward"   : tp_ward,
        "fp_ward"   : fp_ward,
        "fn_ward"   : fn_ward,
        "tn_ward"   : tn_ward,
        "ari_skater": ari_skater,
        "ami_skater": ami_skater,
        "tp_skater" : tp_skater,
        "fp_skater" : fp_skater,
        "fn_skater" : fn_skater,
        "tn_skater" : tn_skater,
    }

In [9]:
for shape, size in itertools.product(shapes, sizes):
    print(f"Evaluating synthetic {shape} - Size: {size}...")

    res_df = pd.read_parquet(f"results/regionalization/synthetic/reg_{shape}_{size}.parquet")
    gdf    = gpd.read_parquet(f"data/regionalization/gdf_{shape}_{size}.parquet")

    true_cols = [c for c in gdf.columns if c.startswith("true_")]
    gdf_truth = gdf[true_cols].copy()
    gdf_truth.index.name = "obs_id"
    gdf_truth = gdf_truth.reset_index()

    merged = res_df.merge(gdf_truth, on="obs_id")

    groups = list(merged.groupby(["method", "missing_edges", "c_run", "z_run"]))

    results = Parallel(n_jobs=-1)(
        delayed(evaluate_group_synthetic)(key, group)
        for key, group in groups
    )

    metrics_df = pd.DataFrame(results)
    output_path = f"summary/regionalization/synthetic/metrics_{shape}_{size}.parquet"
    metrics_df.to_parquet(output_path, index=False)
    print(f"  Saved: {output_path}  |  shape: {metrics_df.shape}\n")

Evaluating synthetic square - Size: 100...
  Saved: summary/regionalization/synthetic/metrics_square_100.parquet  |  shape: (5700, 16)

Evaluating synthetic square - Size: 400...
  Saved: summary/regionalization/synthetic/metrics_square_400.parquet  |  shape: (5700, 16)

Evaluating synthetic pent - Size: 100...
  Saved: summary/regionalization/synthetic/metrics_pent_100.parquet  |  shape: (5700, 16)

Evaluating synthetic pent - Size: 400...
  Saved: summary/regionalization/synthetic/metrics_pent_400.parquet  |  shape: (5700, 16)

Evaluating synthetic hex - Size: 100...
  Saved: summary/regionalization/synthetic/metrics_hex_100.parquet  |  shape: (5700, 16)

Evaluating synthetic hex - Size: 400...
  Saved: summary/regionalization/synthetic/metrics_hex_400.parquet  |  shape: (5700, 16)



In [6]:
def evaluate_group_real(group_key, group_df):

    method, missing_edges, c_run = group_key
    ward_pred   = group_df["ward_label"].values
    skater_pred = group_df["skater_label"].values
    ward_true   = group_df["true_ward"].values
    skater_true = group_df["true_skater"].values

    # ward
    if np.all(ward_pred == -1):
        ari_ward = ami_ward = np.nan
        tp_ward = fp_ward = fn_ward = tn_ward = np.nan
    else:
        valid = (ward_pred != -1) & (ward_true != -1)
        if valid.sum() > 1:
            ari_ward = adjusted_rand_score(ward_true[valid], ward_pred[valid])
            ami_ward = adjusted_mutual_info_score(ward_true[valid], ward_pred[valid])
            tp_ward, fp_ward, fn_ward, tn_ward = pair_confusion_counts(
                ward_true[valid], ward_pred[valid]
            )
        else:
            ari_ward = ami_ward = np.nan
            tp_ward = fp_ward = fn_ward = tn_ward = np.nan

    # skater
    if np.all(skater_pred == -1):
        ari_skater = ami_skater = np.nan
        tp_skater = fp_skater = fn_skater = tn_skater = np.nan
    else:
        valid = (skater_pred != -1) & (skater_true != -1)
        if valid.sum() > 1:
            ari_skater = adjusted_rand_score(skater_true[valid], skater_pred[valid])
            ami_skater = adjusted_mutual_info_score(skater_true[valid], skater_pred[valid])
            tp_skater, fp_skater, fn_skater, tn_skater = pair_confusion_counts(
                skater_true[valid], skater_pred[valid]
            )
        else:
            ari_skater = ami_skater = np.nan
            tp_skater = fp_skater = fn_skater = tn_skater = np.nan

    return {
        "method"    : method,
        "missing_edges" : missing_edges,
        "c_run"     : c_run,
        "ari_ward"  : ari_ward,
        "ami_ward"  : ami_ward,
        "tp_ward"   : tp_ward,
        "fp_ward"   : fp_ward,
        "fn_ward"   : fn_ward,
        "tn_ward"   : tn_ward,
        "ari_skater": ari_skater,
        "ami_skater": ami_skater,
        "tp_skater" : tp_skater,
        "fp_skater" : fp_skater,
        "fn_skater" : fn_skater,
        "tn_skater" : tn_skater,
    }

In [16]:
for gdf_name in gdfs_reg:
    print(f"Evaluating real dataset: {gdf_name}...")

    res_df = pd.read_parquet(f"results/regionalization/real/reg_{gdf_name}.parquet")
    gdf    = gpd.read_parquet(f"data/real/{gdf_name}.parquet")

    gdf_truth = gdf[["true_ward", "true_skater"]].copy()
    gdf_truth.index.name = "obs_id"
    gdf_truth = gdf_truth.reset_index()

    merged = res_df.merge(gdf_truth, on="obs_id")

    groups = list(merged.groupby(["method", "missing_edges", "c_run"]))

    results = Parallel(n_jobs=-1)(
        delayed(evaluate_group_real)(key, group)
        for key, group in groups
    )

    metrics_df = pd.DataFrame(results)
    output_path = f"summary/regionalization/real/metrics_{gdf_name}.parquet"
    metrics_df.to_parquet(output_path, index=False)
    print(f"  Saved: {output_path}  |  shape: {metrics_df.shape}\n")

Evaluating real dataset: south...
  Saved: summary/regionalization/real/metrics_south.parquet  |  shape: (190, 15)

Evaluating real dataset: chicago...
  Saved: summary/regionalization/real/metrics_chicago.parquet  |  shape: (190, 15)

Evaluating real dataset: ceara_zika...
  Saved: summary/regionalization/real/metrics_ceara_zika.parquet  |  shape: (190, 15)



In [18]:
for gdf_name in gdfs_reg:
    print(f"Evaluating real dataset: {gdf_name}...")

    res_df = pd.read_parquet(f"results/regionalization/real/extra/reg_{gdf_name}.parquet")
    gdf    = gpd.read_parquet(f"data/real/{gdf_name}.parquet")

    gdf_truth = gdf[["true_ward", "true_skater"]].copy()
    gdf_truth.index.name = "obs_id"
    gdf_truth = gdf_truth.reset_index()

    merged = res_df.merge(gdf_truth, on="obs_id")

    groups = list(merged.groupby(["method", "missing_edges", "c_run"]))

    results = Parallel(n_jobs=-1)(
        delayed(evaluate_group_real)(key, group)
        for key, group in groups
    )

    metrics_df = pd.DataFrame(results)
    output_path = f"summary/regionalization/real/extra/metrics_{gdf_name}.parquet"
    metrics_df.to_parquet(output_path, index=False)
    print(f"  Saved: {output_path}  |  shape: {metrics_df.shape}\n")

Evaluating real dataset: south...
  Saved: summary/regionalization/real/extra/metrics_south.parquet  |  shape: (500, 15)

Evaluating real dataset: chicago...
  Saved: summary/regionalization/real/extra/metrics_chicago.parquet  |  shape: (500, 15)

Evaluating real dataset: ceara_zika...
  Saved: summary/regionalization/real/extra/metrics_ceara_zika.parquet  |  shape: (500, 15)



In [8]:
res_df = pd.read_parquet(f"results/regionalization/real/extra/reg_south_methods.parquet")
gdf    = gpd.read_parquet(f"data/real/south.parquet")

gdf_truth = gdf[["true_ward", "true_skater"]].copy()
gdf_truth.index.name = "obs_id"
gdf_truth = gdf_truth.reset_index()

merged = res_df.merge(gdf_truth, on="obs_id")

groups = list(merged.groupby(["method", "missing_edges", "c_run"]))

results = Parallel(n_jobs=-1)(
        delayed(evaluate_group_real)(key, group)
        for key, group in groups
    )

metrics_df = pd.DataFrame(results)
output_path = f"summary/regionalization/real/extra/metrics_south_methods.parquet"
metrics_df.to_parquet(output_path, index=False)
print(f"  Saved: {output_path}  |  shape: {metrics_df.shape}\n")

  Saved: summary/regionalization/real/extra/metrics_south_methods.parquet  |  shape: (711, 15)

